# Lecture 06 companion notebook — R

**BIOE 610 Computational Tools · Week 6**

Every example from Lecture 06, plus this week's self-study, as cells you can run, change and break — in lecture order: files → data frames → random draws → the tidyverse → ggplot2 → the figure gallery, with reference sections on control flow, gotchas and error messages at the end. There is a matching Python notebook (`Lecture06_Python_companion.ipynb`) with the same section numbers — keep them open side by side to compare languages.

**How to use this notebook in Positron**

1. Open the file — Positron shows it as a notebook automatically
2. Pick the **R kernel** (top right). If no R kernel is offered, run this once in an R console, then reopen:
   ```r
   install.packages("IRkernel")
   IRkernel::installspec()
   ```
3. Run a cell with **Shift-Enter**. Run them in order the first time; then go back and experiment — *change the numbers, break things on purpose, read the errors*.

## 1 · The libraries this notebook uses

`install.packages()` puts a package on your computer — **once**. `library()` makes it available in *this* session — **every** session ("there is no package called ..." means you never installed it). This whole notebook needs just one load:

In [ ]:
library(tidyverse)   # loads readr, dplyr, tidyr, ggplot2, stringr, purrr, ...
# use one function without loading its package: readxl::read_excel("file.xlsx")
# sections 2-13 are base R and would run even without the tidyverse

# Part 1 · Reading and writing files

## 2 · Reading tabular data

`read.csv()` for comma-separated files, `read.delim()` for tab-separated ones. Both read straight from a URL, so this notebook works wherever you saved it. `check.names = FALSE` keeps the column name `Geno&Micro` exactly as it is in the file.

In [ ]:
stickle <- read.delim("https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Data_Folder/Stickle_RNAseq.tsv",
                      check.names = FALSE)
dim(stickle)          # rows, columns
stickle[1:3, 1:5]     # first rows, first columns

In [ ]:
# The two questions every reader function asks: where are the column
# names (header) and what separates the values (sep)?
# Without check.names, R "fixes" awkward names: Geno&Micro becomes Geno.Micro.
# Try it - read just one row and compare the 4th column name:
names(read.delim("https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Data_Folder/Stickle_RNAseq.tsv", nrows = 1))[4]
names(stickle)[4]     # what we kept with check.names = FALSE

## 3 · Writing tabular data

Write **derived** tables into `results/` — never overwrite anything in `data/` (raw data is read-only). `row.names = FALSE`, or R sneaks an extra unnamed column of row numbers into your file.

In [ ]:
genes  <- grep("^Gene", names(stickle))              # which columns are genes?
totals <- data.frame(Individual = stickle$Individual,
                     Total      = rowSums(stickle[, genes]))
head(totals, 3)

In [ ]:
dir.create("results", showWarnings = FALSE)
write.csv(totals, "results/totals.csv", row.names = FALSE)
write.table(totals, "results/totals.tsv", sep = "\t",
            row.names = FALSE, quote = FALSE)
list.files("results")

## 4 · Saving figures to files

R's pattern is *open a file device → plot → `dev.off()`*. Forget `dev.off()` and the file is empty or locked. Vector formats (`.svg`, `.pdf`) scale forever — use them for papers; raster (`.png`) needs a resolution (`res = 300`).

In [ ]:
dir.create("results", showWarnings = FALSE)   # make sure the folder exists

svg("results/first_figure.svg", width = 7, height = 5)   # open the device
boxplot(totals$Total, ylab = "Total reads per fish")      # draw into it
dev.off()                                                 # close it - ALWAYS

png("results/first_figure.png", width = 7, height = 5, units = "in", res = 300)
boxplot(totals$Total, ylab = "Total reads per fish")
dev.off()

list.files("results")

# Part 2 · Data frames

## 5 · Creating a data frame by hand

Vectors of equal length become columns. A data frame is a *list of equal-length vectors* — one name per column, one type per column.

In [ ]:
hydrogel_concentration <- factor(c("low", "high", "high", "high",
                                   "medium", "medium", "medium", "low"))
compression  <- c(3.4, 3.4, 8.4, 3, 5.6, 8.1, 8.3, 4.5)
conductivity <- c(0, 9.2, 3.8, 5, 5.6, 4.1, 7.1, 5.3)

mydata <- data.frame(hydrogel_concentration, compression, conductivity)
row.names(mydata) <- paste0("Sample_", 1:8)
mydata

## 6 · The stickleback table as a data frame

First questions for any new table: how big (`dim`), what's in it (`head`, `str`), how do the groups break down (`table`), what does one column look like (`summary`).

In [ ]:
dim(stickle)
table(stickle$Genotype, stickle$Microbiota)
summary(stickle$Gene1)

## 7 · Summary statistics by group

`aggregate()`'s formula — *response `~` groups* — is exactly the formula `boxplot()` uses in the next section. The tidyverse's `group_by() |> summarize()` generalizes this - see section 19.

In [ ]:
aggregate(Gene1 ~ Genotype + Microbiota, data = stickle, FUN = median)

In [ ]:
sapply(stickle[, c("Gene1", "Gene2", "Gene3")], mean)

# Part 3 · Random draws from known distributions

### Why draw random numbers? And with which arguments?

A scientist generates data constantly: to **test an analysis** on data where the right answer is known, to **build a null model** (what would the data look like if nothing were going on?), and to **plan experiments** by simulation. Each distribution is a different data-generating machine, with its own parameters as arguments — and the first argument is always `n`, the number of draws:

| Distribution | R | Python (numpy generator) |
|:---|:---|:---|
| Uniform | `runif(n, min = 0, max = 100)` | `rng.uniform(0, 100, n)` |
| Normal | `rnorm(n, mean = 50, sd = 10)` | `rng.normal(50, 10, n)` |
| Poisson | `rpois(n, lambda = 5)` | `rng.poisson(5, n)` |

`set.seed()` (R) / `np.random.default_rng(seed)` (Python) make the draws reproducible: same seed, same "random" numbers.

## 8 · Drawing random samples

The `r*` family — `rnorm()`, `runif()`, `rpois()`, `rbinom()` — draws from named distributions; `sample()` draws from anything you give it. `set.seed()` makes the "random" numbers reproducible.

In [ ]:
set.seed(2026)
x <- rnorm(10000, mean = 0, sd = 10)
y <- sample(1:10000, 10000, replace = TRUE)
plot(x, y, pch = 16, cex = 0.3, col = adjustcolor("#31688e", 0.4))

In [ ]:
# Your turn within a cell: change mean and sd and re-run.
# What does sd = 1 vs sd = 50 do to the cloud?
hist(rnorm(10000, mean = 0, sd = 10), breaks = 40)

## 9 · Histograms and the honest density overlay

`freq = FALSE` puts the bars on the **density** scale, so the theoretical curve `dnorm()` drops straight on top — no magic scaling constants. `curve(..., add = TRUE)` is a *low-level* call: it draws onto the open figure.

In [ ]:
x <- rnorm(1000, mean = 0, sd = 100)
hist(x, breaks = 30, freq = FALSE, col = "grey90", border = "white")
curve(dnorm(x, mean = 0, sd = 100), add = TRUE, col = "#440154", lwd = 2)

## 10 · Three distributions at once — and multi-panel figures

`par(mfrow = c(1, 3))` splits the plotting device into a grid; each high-level call fills the next panel. Three workhorses, as in the lecture: **uniform** (anything equally likely), **normal** (measurements), **Poisson** (counts). In a script you would wrap this in `svg("results/distributions.svg", ...)` ... `dev.off()` to save it — here we keep the figure inline.

In [ ]:
set.seed(2026); n <- 2000
par(mfrow = c(1, 3), mar = c(4, 4, 2, 1))
hist(runif(n, 0, 100),  breaks = 40, col = "#31688e", border = "white",
     main = "Uniform(0, 100)",  xlab = "value")
hist(rnorm(n, 50, 10),  breaks = 40, col = "#440154", border = "white",
     main = "Normal(50, 10)",   xlab = "value")
hist(rpois(n, 5),       breaks = seq(-0.5, 20.5, 1), col = "#35b779",
     border = "white", main = "Poisson(5)",    xlab = "count")
par(mfrow = c(1, 1))     # reset, or every later plot lands in the grid

In [ ]:
# Change n, the parameters and breaks until each panel surprises you.
# What happens to Poisson(5) as you raise lambda to 50?
# Then add a fourth panel: rbinom(n, 20, 0.3) - successes out of 20 tries.
hist(rpois(2000, 50), breaks = 40, col = "#35b779", border = "white")

# Part 4 · Base graphics and first figures (self-study)

## 11 · Base graphics vocabulary

**High-level** functions start a figure (`plot()`, `hist()`, `boxplot()`); **low-level** functions add to the open one (`points()`, `lines()`, `curve(add = TRUE)`, `legend()`). Arguments do the styling: `type=`, `col=`, `pch=`, `lwd=`, `xlab=` / `ylab=` / `main=`.

In [ ]:
seq_1 <- seq(0, 10, by = 0.1)
plot(seq_1, sin(seq_1), type = "l", lwd = 2, col = "#440154",
     xlab = "x", ylab = "sin(x)", main = "plot(), then points(), then legend()")
points(seq_1[seq(1, 101, by = 10)], sin(seq_1)[seq(1, 101, by = 10)],
       pch = 16, col = "#35b779")
legend("topright", legend = c("line", "every 10th point"),
       col = c("#440154", "#35b779"), lty = c(1, NA), pch = c(NA, 16), bty = "n")

## 12 · A first real figure

In [ ]:
boxplot(Gene1 ~ Genotype + Microbiota, data = stickle,
        col = c("steelblue", "tomato", "steelblue", "tomato"),
        ylab = "Gene1 read count", las = 2)

## 13 · The same figure, dressed for a lab meeting

Every improvement is one argument or one extra call — colors, names, a legend, real labels. It works, but it is manual bookkeeping; ggplot2 derives all of this *from the data* - sections 21-23 build that up layer by layer.

In [ ]:
boxplot(Gene1 ~ Microbiota + Genotype, data = stickle,
        col = c("#44015499", "#35b77999"),
        names = c("conv\nA", "mono\nA", "conv\nB", "mono\nB"),
        las = 1, xlab = "", ylab = "Gene1 read count",
        main = "Stickleback gut RNA-seq: Gene1 by group")
legend("topright", fill = c("#44015499", "#35b77999"), title = "Microbiota",
       legend = c("conventional", "mono"), bty = "n")

## 14 · Inspecting objects

When an object surprises you, interrogate it first. These five answer most questions.

In [ ]:
stickle <- read_tsv("https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Data_Folder/Stickle_RNAseq.tsv",
                    show_col_types = FALSE) |>
  rename(geno_micro = `Geno&Micro`)       # fix the awkward name once

class(stickle)
dim(stickle)
names(stickle)[1:6]

In [ ]:
str(stickle[, 1:5])        # structure: type, dimensions, first values
summary(stickle$Gene1)     # quick summary of one column

## 15 · Logical operators and membership

Comparisons return logical vectors; logical vectors select rows. New today: **membership** with `%in%`.

In [ ]:
x <- c(1, 5, 10, 15, 20)
x > 10                  # one TRUE/FALSE per element
x[x > 10]               # keep only where TRUE - L05's logical indexing
x %in% c(1, 15, 99)     # membership: is each element in the set?

## 16 · Control flow: making decisions with `if`

R wraps blocks in `{ }`. The vectorized cousin `ifelse()` gives one answer per element — no loop needed.

In [ ]:
mass <- 1.42
if (mass > 1.0) {
  cat("big fish\n")
} else if (mass > 0.5) {
  cat("medium fish\n")
} else {
  cat("small fish\n")
}

In [ ]:
x <- c(0.3, 0.8, 1.42)
ifelse(x > 1.0, "big", "small")      # vectorized: one answer per element

## 17 · Control flow: repeating with `for` (and `while`)

Both languages loop directly over a collection's *elements* — no counters required. `break` leaves early; `next` skips to the next turn.

In [ ]:
genes <- c("tp53", "brca1", "myc")
for (g in genes) {
  cat(toupper(g), "\n")
}

for (i in seq_along(genes)) {        # ... or over the positions 1, 2, 3
  cat(i, genes[i], "\n")
}

In [ ]:
i <- 1
while (i < 8) {                      # repeat until the condition fails
  i <- i * 2
}
i

## 18 · When you do *not* need a loop

Vectorized code is shorter, faster and easier to read. Loops earn their keep for files, simulations, and step-*n*-depends-on-step-*n−1* problems.

In [ ]:
x <- c(2, 3, 4, 5)
x^2                                          # not: for (i in seq_along(x)) ...
genes_cols <- grep("^Gene", names(stickle))
head(rowSums(stickle[, genes_cols]))         # not: nested loops

## 19 · The pipe `|>`: "and then"

Each verb takes a data frame and returns a data frame, so they chain. Read `|>` as **"and then"**.

In [ ]:
mydata <- tibble(
  hydrogel_concentration = factor(c("low", "high", "high", "high",
                                    "medium", "medium", "medium", "low")),
  compression  = c(3.4, 3.4, 8.4, 3, 5.6, 8.1, 8.3, 4.5),
  conductivity = c(0, 9.2, 3.8, 5, 5.6, 4.1, 7.1, 5.3))

mydata |>
  filter(compression > 4) |>                         # keep rows
  mutate(ratio = conductivity / compression) |>      # add a column
  group_by(hydrogel_concentration) |>                # define groups
  summarize(n = n(), mean_ratio = mean(ratio))       # one row per group

## 20 · Wide to long: `pivot_longer()`

One `pivot_longer()` unlocks *every* gene at once: 80 × 600 = 48,000 rows, one per fish × gene.

In [ ]:
stickle_long <- stickle |>
  pivot_longer(starts_with("Gene"), names_to = "gene", values_to = "count")
stickle_long

In [ ]:
gene_means <- stickle_long |>
  group_by(gene, Genotype, Microbiota) |>
  summarize(mean_count = mean(count), .groups = "drop")
gene_means |> filter(gene == "Gene1")

## 21 · ggplot2, layer 1: data + aesthetics + a geom

`ggplot(data, aes(...))` declares the **mapping**; a **geom** draws it. `geom_jitter()` is `geom_point()` plus sideways noise.

In [ ]:
gene1 <- stickle_long |> filter(gene == "Gene1")

ggplot(gene1, aes(x = Genotype, y = count)) +
  geom_jitter(width = 0.08)

## 22 · Layer 2: more aesthetics, more geoms

Mapping a column to `fill` / `color` splits the groups **and** builds the legend — automatically, from the data. Layers draw in order.

In [ ]:
ggplot(gene1, aes(x = Genotype, y = count, fill = Microbiota)) +
  geom_violin(alpha = 0.4, color = NA, position = position_dodge(0.8)) +
  geom_point(aes(color = Microbiota), alpha = 0.6,
             position = position_jitterdodge(jitter.width = 0.08, dodge.width = 0.8))

## 23 · Layers 3–5: facets, scales, labels, theme — and `ggsave()`

`facet_wrap(~ gene)` gives one panel per gene for free. This is the left half of L06's destination figure.

In [ ]:
stickle_long |>
  filter(gene %in% paste0("Gene", 1:4)) |>
  ggplot(aes(Genotype, count, fill = Microbiota)) +
  geom_violin(alpha = 0.4, color = NA, position = position_dodge(0.8)) +
  geom_point(aes(color = Microbiota), alpha = 0.6, size = 1.2,
             position = position_jitterdodge(jitter.width = 0.08, dodge.width = 0.8)) +
  scale_fill_viridis_d(end = 0.7) + scale_color_viridis_d(end = 0.7) +
  facet_wrap(~ gene, scales = "free_y") +
  labs(x = "Host genotype", y = "Read count",
       title = "Stickleback gut RNA-seq",
       subtitle = "Each point is one fish; violins show the full distribution") +
  theme_minimal(base_size = 14) + theme(legend.position = "top")

In [ ]:
dir.create("results", showWarnings = FALSE)
ggsave("results/gene_panel.svg", width = 8, height = 6)   # saves the last plot drawn
list.files("results")

## 24 · Six geoms, one example each

The lecture's geom tour as runnable cells — histogram, boxplot, scatter, trend, line and bars, all from the stickleback table. Each ends with the save-line convention: `ggsave()` into `results/`.

In [ ]:
dir.create("results", showWarnings = FALSE)   # for the save lines below

# geom_histogram(): one continuous column; the geom does the binning
ggplot(gene1, aes(x = count)) +
  geom_histogram(bins = 30, fill = "#31688e", color = "white") +
  labs(x = "Gene1 read count", y = "Number of fish")
ggsave("results/gene1_histogram.svg", width = 7, height = 5)

In [ ]:
# geom_boxplot(): continuous by category; a second category dodges via fill
stickle_long |>
  filter(gene == "Gene2") |>
  ggplot(aes(x = Genotype, y = count, fill = Microbiota)) +
  geom_boxplot(alpha = 0.7) +
  scale_fill_viridis_d(end = 0.7) +
  labs(x = "Host genotype", y = "Gene2 read count")
ggsave("results/gene2_boxplot.svg", width = 7, height = 5)

In [ ]:
# geom_point(): two continuous columns; a third rides along as color
ggplot(stickle, aes(x = Gene1, y = Gene2, color = Genotype)) +
  geom_point(size = 2.5, alpha = 0.7) +
  scale_color_viridis_d(end = 0.6) +
  labs(x = "Gene1 read count", y = "Gene2 read count")
ggsave("results/gene1_gene2_scatter.svg", width = 7, height = 5)

In [ ]:
# geom_smooth(): a trend is just another layer (method = "lm" fits a line)
ggplot(stickle, aes(x = Gene1, y = Gene2)) +
  geom_point(alpha = 0.5, color = "#31688e") +
  geom_smooth(method = "lm", color = "#440154") +
  labs(x = "Gene1 read count", y = "Gene2 read count")
ggsave("results/gene1_gene2_trend.svg", width = 7, height = 5)

In [ ]:
# geom_line(): lines need an ORDERED x - here, genes ranked by mean expression
stickle_long |>
  group_by(gene) |>
  summarize(mean_count = mean(count)) |>
  arrange(desc(mean_count)) |>
  mutate(rank = row_number()) |>
  ggplot(aes(x = rank, y = mean_count)) +
  geom_line(color = "#440154", linewidth = 1) +
  scale_y_log10() +
  labs(x = "Gene rank (by mean expression)", y = "Mean read count (log)")
ggsave("results/gene_rank_means.svg", width = 7, height = 5)

In [ ]:
# geom_col(): plots values YOU computed (geom_bar counts rows for you)
gene1 |>
  group_by(Genotype, Microbiota) |>
  summarize(median_count = median(count), .groups = "drop") |>
  ggplot(aes(x = Genotype, y = median_count, fill = Microbiota)) +
  geom_col(position = "dodge") +
  scale_fill_viridis_d(end = 0.7) +
  labs(x = "Host genotype", y = "Median Gene1 read count")
ggsave("results/gene1_group_medians.svg", width = 7, height = 5)

## 25 · The closer: all 600 genes in one figure

`group_by(gene) |> summarize()` collapses 48,000 rows to 600 — one point per gene. Dashed line: variance = mean (Poisson); real RNA-seq runs hotter.

In [ ]:
stickle_long |>
  group_by(gene) |>
  summarize(mean = mean(count), var = var(count)) |>
  ggplot(aes(mean, var)) +
  geom_point(alpha = 0.4, color = "#31688e") +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed") +
  geom_smooth(se = FALSE, color = "#440154") +
  scale_x_log10() + scale_y_log10() +
  labs(x = "Mean count per gene (log)", y = "Variance (log)",
       title = "All 600 genes at once") +
  theme_minimal(base_size = 14)

# The figure gallery

## 26 · A clustered heatmap, from synthesized expression data

`scale = "row"` standardizes each gene, so the colors compare *patterns*, not levels. The dendrograms should rediscover the two modules and the control/treated split we build in.

In [ ]:
set.seed(2026)
expr <- matrix(rnorm(30 * 12, mean = 8, sd = 0.4), nrow = 30) +
        rnorm(30)                                 # a baseline level per gene
expr[1:15, 7:12]  <- expr[1:15, 7:12] + 2         # module 1: up in treated
expr[16:30, 7:12] <- expr[16:30, 7:12] - 1.5      # module 2: down in treated
rownames(expr) <- paste0("gene_", 1:30)
colnames(expr) <- c(paste0("ctrl_", 1:6), paste0("treat_", 1:6))

heatmap(expr, scale = "row", col = hcl.colors(50, "viridis"), margins = c(6, 5))

## 27 · Growth curves with ribbons

The `group_by() |> summarize()` + ribbon pattern works for any repeated time course: sensor logs, qPCR, cyclic loading.

In [ ]:
params <- tibble(strain = c("wild type", "mutant A", "mutant B"),
                 K = c(1.8, 1.2, 1.6),            # carrying capacity
                 r = c(0.25, 0.35, 0.15))         # growth rate
growth <- expand_grid(params, rep = 1:4, time = seq(0, 48, by = 2)) |>
  mutate(od = K / (1 + exp(-r * (time - 20))) + rnorm(n(), sd = 0.05))

growth |>
  group_by(strain, time) |>
  summarize(mean = mean(od), sd = sd(od), .groups = "drop") |>
  ggplot(aes(time, mean, color = strain, fill = strain)) +
  geom_ribbon(aes(ymin = mean - sd, ymax = mean + sd), alpha = 0.25, color = NA) +
  geom_line(linewidth = 1) +
  scale_color_viridis_d(end = 0.8) + scale_fill_viridis_d(end = 0.8) +
  labs(x = "Time (h)", y = "OD600",
       title = "Bioreactor growth curves (mean ± SD of 4 replicates)") +
  theme_minimal(base_size = 14)

## 28 · Dose-response curves

The log-scaled x axis is what makes dose-response curves readable — doses span five orders of magnitude. The IC50 is where each curve crosses 50%.

In [ ]:
dr <- expand_grid(tibble(compound = c("compound X", "compound Y"),
                         ic50 = c(3e-7, 5e-6)),
                  rep = 1:3,
                  dose = 10^seq(-9, -4, length.out = 8)) |>
  mutate(viability = 100 / (1 + (dose / ic50)^1.2) + rnorm(n(), sd = 4))

ggplot(dr, aes(dose, viability, color = compound)) +
  geom_point(alpha = 0.7, size = 2) +
  geom_smooth(se = FALSE) +
  scale_x_log10() +
  scale_color_viridis_d(end = 0.6) +
  labs(x = "Dose (M)", y = "Viability (%)",
       title = "Dose-response: two compounds, three replicates") +
  theme_minimal(base_size = 14)

## 29 · PCA of the real stickleback fish

PCA compresses 600 correlated gene columns into a few axes of maximal variation. `log1p()` tames the counts; `scale. = TRUE` weights every gene equally.

In [ ]:
gene_mat <- stickle |> select(starts_with("Gene"))
pca <- prcomp(log1p(gene_mat), scale. = TRUE)
summary(pca)$importance[, 1:3]           # variance explained by PC1-PC3

In [ ]:
bind_cols(as_tibble(pca$x[, 1:2]),
          stickle |> select(geno_micro)) |>
  ggplot(aes(PC1, PC2, color = geno_micro)) +
  geom_point(size = 2.5, alpha = 0.8) +
  scale_color_viridis_d(name = "Group") +
  labs(title = "PCA: 80 fish from 600 genes",
       subtitle = "Distance between points = overall expression similarity") +
  theme_minimal(base_size = 14)

## 30 · Common gotchas (reference)

Return here when something bites:

- **R counts from 1**, and `x[-1]` *drops* the first element (it is not "the last" as in Python)
- **`=` vs `==`**: `=` assigns or names an argument; `==` tests equality
- **`NA` is contagious**: `mean(c(1, NA, 3))` is `NA` - use `na.rm = TRUE`
- **Floating point**: `0.1 + 0.2 == 0.3` is `FALSE` - use `all.equal()`
- **Case matters**: `Data` and `data` are different objects
- **Working directory**: "cannot open file" usually means R is looking in the wrong folder - check `getwd()` and use project-relative paths
- **Factors in disguise**: `as.numeric()` on a factor returns the level *codes*, not the labels - use `as.numeric(as.character(x))`

## 31 · Reading error messages (reference)

| Error message | Meaning | Fix |
|:---|:---|:---|
| `object 'x' not found` | Variable doesn't exist | Check spelling; run the line that creates it |
| `could not find function` | Package not loaded | `library()` the package, or check spelling |
| `unexpected ')'` / `unexpected symbol` | Mismatched brackets or quotes | Count opening/closing brackets |
| `non-numeric argument to binary operator` | Wrong data type | Check with `class()` or `str()` |
| `cannot open file ... No such file` | Wrong working directory or path | `getwd()`, `list.files()`, fix the path |

R reports the failing call. When stuck, paste the *exact* message into a search engine or an AI assistant - then verify the answer against the documentation.

## Your turn

1. **Script the summaries**: write `scripts/summarize_stickleback.R` or `.py` that reads the data, writes a per-group median table for a gene of your choice to `results/`, and saves the distribution gallery as SVG *and* a 300-dpi PNG
2. **Rebuild Example 1** (the section-23 panel) for four *different* genes — in the language you know less well — and save it into `results/`
3. Color the mean-variance plot (section 25): compute each gene's mean separately for genotypes A and B, and color each point by which genotype is higher
4. **Stretch**: adapt the gallery heatmap to the real data — the 30 most *variable* stickleback genes, clustering the 80 fish (`scale = "row"` matters!)
5. Change the distribution gallery (section 10) until each panel surprises you — then add a fourth panel: the binomial

Working versions of the two scripts are on the Examples page — but only look after your own runs.